# Single trajectory

A short Monte Carlo run on chignolin (PDB 1UAO), start to finish: build a
force field, run sampling, and look at where the energy went.

Two conventions to keep in mind throughout:

- **Temperature is dimensionless.** It is a reduced parameter, roughly 0.3
  (cold, folded) to 0.6 (hot, unfolded). It is not in Kelvin.
- **Energies are unitless.** They are sums of knowledge-based table entries
  scaled by a dimensionless per-group weight. There is no Boltzmann constant
  anywhere in the engine.


## Build the system

`MCPUForceField` is the bridge from an mdtraj topology to a simulatable
system. It assigns atom types, reorders atoms into the contiguous layout the
engine requires, and loads the fitted potentials.

The engine is heavy-atom only, so strip hydrogens first — the H-bond term
builds its own virtual amide hydrogens where it needs them.


In [ ]:
import numpy as np
import mdtraj as md

import pymcpu as mc
from pymcpu.forcefields.mcpu import MCPUForceField
from pymcpu.runners import default_example_pdb

traj = md.load(str(default_example_pdb()))
heavy = traj.atom_slice(traj.topology.select("not element H"))

forcefield = MCPUForceField(heavy, param_set="mcpu08")
system = forcefield.create_system(heavy.topology)

print(f"{system.get_num_atoms()} atoms, {system.get_num_residues()} residues")

## Set up the integrator and simulation

`Integrator` holds the Monte Carlo move parameters. Set the seed explicitly:
the whole trajectory is determined by it, so a run is reproducible only if you
record it.

`Integrator` defaults to `temperature=300.0`, which is *not* a reduced
temperature — always pass one.


In [ ]:
integrator = mc.Integrator(temperature=0.6, step_size_rad=0.1)
integrator.set_seed(42)

sim = mc.Simulation(heavy.topology, system, integrator)
sim.context.set_positions((forcefield.coords[0] * 10.0).T.astype(np.float32))

print(sim.describe())

## Look at the starting energy

`energy_breakdown()` reports both the raw table sums and the weighted values.
`by_name` holds the energy of each term, labelled with the names the force
field gave them (`system.energy_terms()`).

In [ ]:
breakdown = sim.context.energy_breakdown(weighted=True)

for name, value in breakdown["by_name"].items():
    print(f"  {name:20} {value:12.4f}")
print(f"  {'weighted total':20} {breakdown['weighted_total']:12.4f}")

## Run

Attach an `EnergyReporter` before stepping. It writes a CSV row every
`interval` steps with the per-group energies and the cumulative move
accept/attempt counters. Reporters write files rather than printing, because
under replica exchange stdout from many ranks interleaves into noise.


In [ ]:
sim.add_energy_reporter("energies.csv", interval=100)
sim.step(10_000)

print(f"completed {sim.current_step} steps")

## Acceptance rates

Acceptance rates are the first thing to check on any MC run. Very low rates
mean the step size is too large for this temperature; very high rates mean the
moves are too timid to explore.


In [ ]:
for kind, (accepted, attempted) in integrator.move_counts().items():
    rate = accepted / attempted if attempted else float("nan")
    print(f"  {kind:12} {accepted:6d} / {attempted:6d}   {rate:6.1%}")

## Read the trajectory of energies back

The reporter is a writer, not a buffer — there is no accessor to read
energies out of it. Read the CSV it produced.


In [ ]:
import pandas as pd

energies = pd.read_csv("energies.csv")
print(energies[["step", "total", "mu", "hydrogen_bond"]].iloc[::20].to_string(index=False))

## Where to go next

- Replica exchange, serial and MPI: see the *Running REMD* guide.
- Surviving a scheduler timeout: see *Checkpointing and resume*.
- What each energy term actually computes: see the *Physics Background*
  chapters.
